# 재활용 이미지 VQA 대회용 최종 노트북

이 노트북은 **baseline_colab.ipynb 형식**을 유지하면서, 실제 대회 제출에 맞게 다음 항목을 강화한 최종 버전입니다.

- **모델**: `Qwen/Qwen2.5-VL-7B-Instruct` + **QLoRA**
- **데이터 병합**: `train.csv` + `human_check.csv` + `dev.csv(최빈값)`
- **학습 라벨 방식 개선**: 알파벳이 아니라 **실제 정답 텍스트**를 학습
- **추론 후처리 개선**: 생성된 텍스트를 `a,b,c,d` 보기와 다시 매칭하여 **제출용 알파벳** 출력
- **경로 오류 대응**: `train / trian / dev / test` 폴더명 차이 자동 대응
- **빠른 실험 모드**: 주석 해제만 하면 **500개 샘플**로 먼저 테스트 가능

Colab Pro+ / A100 환경을 기준으로 작성했습니다.


# 환경 준비

개발 환경에 필요한 라이브러리를 설치합니다.

- 아래 셀 실행
- 실행 완료 후 **런타임 → 세션 다시 시작**


In [ ]:
!pip -q install -U git+https://github.com/huggingface/transformers accelerate
!pip -q install -U bitsandbytes peft sentencepiece Pillow tqdm pandas scikit-learn

# GPU / 라이브러리 버전 확인

In [ ]:
import torch, transformers, peft, platform
print("Python:", platform.python_version())
print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)


# 데이터 준비

구글 드라이브를 마운트하고, CSV와 이미지 폴더 경로를 설정합니다.

사용 경로 기준:
- `MyDrive/Colab Notebooks/content/`
- CSV: `train.csv`, `dev.csv`, `human_check.csv`, `test.csv`, `sample_submission.csv`
- 이미지 폴더: `train/`, `dev/`, `test/`
- 폴더명이 `trian/`으로 잘못 저장된 경우도 자동 탐지


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path

# =========================
# 기본 경로
# =========================
BASE_DIR = "/content/drive/MyDrive/Colab Notebooks/content"

TRAIN_CSV = os.path.join(BASE_DIR, "train.csv")
DEV_CSV = os.path.join(BASE_DIR, "dev.csv")
HC_CSV = os.path.join(BASE_DIR, "human_check.csv")
TEST_CSV = os.path.join(BASE_DIR, "test.csv")
SAMPLE_SUB_CSV = os.path.join(BASE_DIR, "sample_submission.csv")

OUTPUT_DIR = os.path.join(BASE_DIR, "qwen25vl_recycling_final_ckpt")
os.makedirs(OUTPUT_DIR, exist_ok=True)

# =========================
# 이미지 폴더 자동 탐지
# =========================
def find_split_dir(base_dir, split_name):
    candidates = [
        os.path.join(base_dir, split_name),
        os.path.join(base_dir, split_name.lower()),
        os.path.join(base_dir, split_name.upper()),
    ]
    # train 오타 대응
    if split_name == "train":
        candidates += [
            os.path.join(base_dir, "trian"),
            os.path.join(base_dir, "TriAN"),
        ]
    for c in candidates:
        if os.path.isdir(c):
            return c
    return None

TRAIN_DIR = find_split_dir(BASE_DIR, "train")
DEV_DIR = find_split_dir(BASE_DIR, "dev")
TEST_DIR = find_split_dir(BASE_DIR, "test")

print("BASE_DIR :", BASE_DIR)
print("TRAIN_DIR:", TRAIN_DIR)
print("DEV_DIR  :", DEV_DIR)
print("TEST_DIR :", TEST_DIR)

for p in [TRAIN_CSV, DEV_CSV, HC_CSV, TEST_CSV, SAMPLE_SUB_CSV]:
    print(os.path.basename(p), "->", os.path.exists(p))


# 라이브러리, 설정, 시드 고정

A100 환경에서 안정적으로 돌아가도록 기본 하이퍼파라미터를 설정합니다.

아래 **QUICK_MODE** 주석을 해제하면 500개 샘플로 먼저 빠르게 검증할 수 있습니다.


In [ ]:
import os, re, math, random, difflib, gc
import pandas as pd
import numpy as np
from PIL import Image
from dataclasses import dataclass
from typing import Any, Dict, List
from collections import Counter
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

from transformers import (
    AutoProcessor,
    Qwen2_5_VLForConditionalGeneration,
    BitsAndBytesConfig,
    get_cosine_schedule_with_warmup,
)
from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
)

Image.MAX_IMAGE_PIXELS = None
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

# =========================
# 전역 설정
# =========================
MODEL_ID = "Qwen/Qwen2.5-VL-7B-Instruct"
IMAGE_SIZE = 448
MAX_NEW_TOKENS = 12
SEED = 42

NUM_EPOCHS = 2
LR = 2e-4
WEIGHT_DECAY = 0.01
BATCH_SIZE = 1
GRAD_ACCUM = 8
WARMUP_RATIO = 0.05
MAX_GRAD_NORM = 1.0

LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05

# QUICK_MODE = True
QUICK_MODE = False
QUICK_N = 500

# 검증 비율
VALID_SIZE = 0.05

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("MODEL_ID =", MODEL_ID)
print("IMAGE_SIZE =", IMAGE_SIZE)
print("QUICK_MODE =", QUICK_MODE)


# 데이터 로드 및 병합

학습 데이터는 다음 순서로 구성합니다.

1. `train.csv`: 기본 학습 데이터
2. `human_check.csv`: 사람이 직접 검수한 고신뢰 데이터
3. `dev.csv`: `answer1~answer5`의 **최빈값**으로 정답 생성  
   단, `human_check`에 포함된 샘플은 제외

또한 학습용 라벨은 알파벳이 아니라 **실제 보기 텍스트**로 변환합니다.

예:
- `answer = b`
- `b = "유리 병"`
- 최종 학습 라벨 = `"유리 병"`


In [ ]:
def safe_read_csv(path):
    df = pd.read_csv(path)
    print(f"{os.path.basename(path)} shape =", df.shape)
    return df

train_df = safe_read_csv(TRAIN_CSV)
dev_df = safe_read_csv(DEV_CSV)
hc_df = safe_read_csv(HC_CSV)
test_df = safe_read_csv(TEST_CSV)
sample_sub_df = safe_read_csv(SAMPLE_SUB_CSV)

ANSWER_COLS = ["answer1", "answer2", "answer3", "answer4", "answer5"]
CHOICE_COLS = ["a", "b", "c", "d"]
VALID_CHOICES = set(CHOICE_COLS)

def majority_vote(row):
    votes = [str(row[c]).strip().lower() for c in ANSWER_COLS if pd.notna(row[c])]
    votes = [v for v in votes if v in VALID_CHOICES]
    if not votes:
        return None
    cnt = Counter(votes)
    max_cnt = max(cnt.values())
    tied = {k for k, v in cnt.items() if v == max_cnt}
    # 동점이면 answer1~5 순서대로 먼저 등장한 값 사용
    for v in votes:
        if v in tied:
            return v
    return votes[0]

def letter_to_text(row, letter_col="answer"):
    letter = str(row[letter_col]).strip().lower()
    if letter not in VALID_CHOICES:
        return None
    return str(row[letter]).strip()

# -------------------------
# 절대 경로 보정
# -------------------------
def resolve_image_path(path_value, split_hint=None):
    p = str(path_value).strip().replace("\\", "/")
    basename = os.path.basename(p)

    candidates = []
    if os.path.isabs(p):
        candidates.append(p)
    else:
        candidates.append(os.path.join(BASE_DIR, p))

    if split_hint == "train":
        if TRAIN_DIR: candidates.append(os.path.join(TRAIN_DIR, basename))
    elif split_hint == "dev":
        if DEV_DIR: candidates.append(os.path.join(DEV_DIR, basename))
    elif split_hint == "test":
        if TEST_DIR: candidates.append(os.path.join(TEST_DIR, basename))

    # path 문자열 자체에 split 정보가 들어있는 경우도 대응
    if "train" in p or "trian" in p:
        if TRAIN_DIR: candidates.append(os.path.join(TRAIN_DIR, basename))
    if "dev" in p:
        if DEV_DIR: candidates.append(os.path.join(DEV_DIR, basename))
    if "test" in p:
        if TEST_DIR: candidates.append(os.path.join(TEST_DIR, basename))

    for c in candidates:
        if c and os.path.exists(c):
            return c
    return candidates[0] if candidates else p

train_df["image_path"] = train_df["path"].apply(lambda x: resolve_image_path(x, "train"))
dev_df["image_path"] = dev_df["path"].apply(lambda x: resolve_image_path(x, "dev"))
hc_df["image_path"] = hc_df["path"].apply(lambda x: resolve_image_path(x, "dev"))
test_df["image_path"] = test_df["path"].apply(lambda x: resolve_image_path(x, "test"))

# -------------------------
# train 정답 텍스트화
# -------------------------
train_df["answer"] = train_df["answer"].astype(str).str.lower().str.strip()
train_df["answer_text"] = train_df.apply(letter_to_text, axis=1)

# -------------------------
# human_check 정답 사용
# -------------------------
hc_df["answer"] = hc_df["answer"].astype(str).str.lower().str.strip()
hc_df["answer_text"] = hc_df.apply(letter_to_text, axis=1)

# -------------------------
# dev 최빈값 생성
# -------------------------
dev_df["answer"] = dev_df.apply(majority_vote, axis=1)
dev_df["answer_text"] = dev_df.apply(letter_to_text, axis=1)

# human_check 우선
hc_ids = set(hc_df["id"].astype(str))
dev_only_df = dev_df[~dev_df["id"].astype(str).isin(hc_ids)].copy()

train_keep = ["id", "path", "image_path", "question", "a", "b", "c", "d", "answer", "answer_text"]
train_final = train_df[train_keep].copy()
hc_final = hc_df[train_keep].copy()
dev_final = dev_only_df[train_keep].copy()

all_train_df = pd.concat([train_final, hc_final, dev_final], ignore_index=True)

# 결측 제거
all_train_df = all_train_df.dropna(subset=["image_path", "question", "a", "b", "c", "d", "answer", "answer_text"]).copy()

# 실제 파일 존재 확인
all_train_df = all_train_df[all_train_df["image_path"].apply(os.path.exists)].reset_index(drop=True)
test_df = test_df[test_df["image_path"].apply(os.path.exists)].reset_index(drop=True)

if QUICK_MODE:
    all_train_df = all_train_df.sample(min(QUICK_N, len(all_train_df)), random_state=SEED).reset_index(drop=True)
    test_df = test_df.head(min(QUICK_N, len(test_df))).reset_index(drop=True)

print("train only       :", len(train_final))
print("human_check      :", len(hc_final))
print("dev majority only:", len(dev_final))
print("-" * 40)
print("merged train size:", len(all_train_df))
print("test size        :", len(test_df))
print("\nanswer(letter) distribution:")
print(all_train_df["answer"].value_counts().sort_index())
print("\nexample rows:")
display(all_train_df.head(3))


# 프롬프트 템플릿

baseline의 객관식 구조는 유지하되, 재활용 데이터 특성에 맞게 다음 내용을 추가합니다.

- 이미지에 **주변 환경이 함께 보일 수 있음**
- 질문이 **재질 / 종류 / 수량 / 색상**을 묻는 경우가 많음
- 최종 학습은 **실제 정답 텍스트**를 출력하도록 구성
- 추론 시에는 생성된 텍스트를 보기와 다시 매칭해 **알파벳**으로 변환


In [ ]:
SYSTEM_INSTRUCT = (
    "당신은 재활용 이미지 질의응답 전문가입니다. "
    "사진 속 재활용품의 재질, 종류, 개수, 색상, 주변 배치까지 꼼꼼히 확인한 뒤 "
    "가장 적절한 보기를 선택해야 합니다. "
    "학습 시에는 정답의 실제 보기 텍스트만 답합니다."
)

def build_user_prompt(question: str, a: str, b: str, c: str, d: str) -> str:
    return (
        "이미지를 자세히 보고 질문에 답하세요.\n"
        "사진에는 주변 환경이 함께 포함될 수 있으므로, 실제 재활용 대상만 구분해야 합니다.\n"
        "질문이 재질, 종류, 수량, 색상 중 무엇을 묻는지 먼저 파악하세요.\n\n"
        f"[질문]\n{question}\n\n"
        f"[보기]\n"
        f"a. {a}\n"
        f"b. {b}\n"
        f"c. {c}\n"
        f"d. {d}\n\n"
        "정답에 해당하는 보기의 텍스트만 그대로 출력하세요. 설명은 쓰지 마세요."
    )

sample_prompt = build_user_prompt(
    "사진 속 재활용 가능한 병의 종류는 무엇인가요?",
    "금속 캔", "유리 병", "플라스틱 병", "종이 팩"
)
print(sample_prompt)


# 모델 / Processor 로드

- `Qwen2.5-VL-7B-Instruct`
- **4-bit NF4 양자화**
- **QLoRA**
- A100에서 안정적으로 학습할 수 있도록 설정


In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    min_pixels=IMAGE_SIZE * IMAGE_SIZE,
    max_pixels=IMAGE_SIZE * IMAGE_SIZE,
    trust_remote_code=True,
)

base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)

base_model = prepare_model_for_kbit_training(base_model)
base_model.gradient_checkpointing_enable()

lora_config = LoraConfig(
    r=LORA_R,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias="none",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    task_type="CAUSAL_LM",
)

model = get_peft_model(base_model, lora_config)
model.print_trainable_parameters()


# Custom Dataset / Collator

이번 최종 버전의 핵심 차이점은 **라벨을 알파벳이 아니라 실제 정답 텍스트로 학습**한다는 점입니다.

예:
- 질문: 사진 속 재활용 가능한 병의 종류는 무엇인가요?
- 보기: a 금속 캔 / b 유리 병 / c 플라스틱 병 / d 종이 팩
- gold label: `유리 병`

또한 loss는 **assistant 응답 부분만 계산**하도록 prompt 구간을 마스킹합니다.


In [ ]:
class RecyclingVQADataset(Dataset):
    def __init__(self, df, train=True):
        self.df = df.reset_index(drop=True)
        self.train = train

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        image = Image.open(row["image_path"]).convert("RGB")

        user_text = build_user_prompt(
            str(row["question"]),
            str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
        )

        prompt_messages = [
            {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
            {"role": "user", "content": [
                {"type": "image", "image": image},
                {"type": "text", "text": user_text}
            ]},
        ]

        item = {
            "id": str(row["id"]),
            "image": image,
            "prompt_messages": prompt_messages,
            "options": {
                "a": str(row["a"]),
                "b": str(row["b"]),
                "c": str(row["c"]),
                "d": str(row["d"]),
            }
        }

        if self.train:
            item["answer"] = str(row["answer"]).strip().lower()
            item["answer_text"] = str(row["answer_text"]).strip()

        return item


@dataclass
class TrainCollator:
    processor: Any

    def __call__(self, batch):
        assert len(batch) == 1, "이 노트북은 안정성을 위해 batch_size=1 기준으로 작성되었습니다."
        sample = batch[0]

        prompt_messages = sample["prompt_messages"]
        full_messages = prompt_messages + [
            {"role": "assistant", "content": [{"type": "text", "text": sample["answer_text"]}]}
        ]

        prompt_text = self.processor.apply_chat_template(
            prompt_messages, tokenize=False, add_generation_prompt=False
        )
        full_text = self.processor.apply_chat_template(
            full_messages, tokenize=False, add_generation_prompt=False
        )

        image = sample["image"]

        enc_full = self.processor(
            text=[full_text],
            images=[image],
            padding=True,
            return_tensors="pt"
        )
        enc_prompt = self.processor(
            text=[prompt_text],
            images=[image],
            padding=True,
            return_tensors="pt"
        )

        labels = enc_full["input_ids"].clone()
        prompt_len = enc_prompt["input_ids"].shape[1]
        labels[:, :prompt_len] = -100

        pad_token_id = self.processor.tokenizer.pad_token_id
        if pad_token_id is not None:
            labels[labels == pad_token_id] = -100

        enc_full["labels"] = labels
        return enc_full


@dataclass
class InferenceCollator:
    processor: Any

    def __call__(self, batch):
        assert len(batch) == 1
        sample = batch[0]
        prompt_text = self.processor.apply_chat_template(
            sample["prompt_messages"],
            tokenize=False,
            add_generation_prompt=True
        )
        enc = self.processor(
            text=[prompt_text],
            images=[sample["image"]],
            padding=True,
            return_tensors="pt"
        )
        return enc, sample


# 학습 / 검증 분리

- 전체 병합 데이터 기준으로 분리
- stratify는 정답 알파벳 기준으로 적용
- 빠른 실험 모드일 때도 동일하게 동작


In [ ]:
train_part, valid_part = train_test_split(
    all_train_df,
    test_size=VALID_SIZE,
    random_state=SEED,
    stratify=all_train_df["answer"]
)

train_part = train_part.reset_index(drop=True)
valid_part = valid_part.reset_index(drop=True)

train_ds = RecyclingVQADataset(train_part, train=True)
valid_ds = RecyclingVQADataset(valid_part, train=True)
test_ds = RecyclingVQADataset(test_df, train=False)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=TrainCollator(processor),
)
valid_loader = DataLoader(
    valid_ds,
    batch_size=1,
    shuffle=False,
    num_workers=0,
    collate_fn=TrainCollator(processor),
)

print("train size:", len(train_ds))
print("valid size:", len(valid_ds))
print("test size :", len(test_ds))


# Fine-tuning

학습 안정성을 위해 다음을 적용합니다.

- **AdamW**
- **Cosine scheduler**
- **Warmup**
- **Gradient Accumulation**
- **bfloat16 autocast**
- **Gradient clipping**
- 검증 loss가 가장 낮을 때 checkpoint 저장


In [ ]:
model.train()

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LR,
    weight_decay=WEIGHT_DECAY
)

num_update_steps_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
num_training_steps = NUM_EPOCHS * num_update_steps_per_epoch
num_warmup_steps = int(num_training_steps * WARMUP_RATIO)

scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)

best_val_loss = float("inf")
global_step = 0

for epoch in range(NUM_EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)
    train_loss_sum = 0.0

    pbar = tqdm(train_loader, total=len(train_loader), desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [train]")
    for step, batch in enumerate(pbar, start=1):
        batch = {k: v.to(device) for k, v in batch.items()}

        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=torch.cuda.is_available()):
            outputs = model(**batch)
            loss = outputs.loss / GRAD_ACCUM

        loss.backward()
        train_loss_sum += loss.item()

        if step % GRAD_ACCUM == 0 or step == len(train_loader):
            torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
            optimizer.step()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            global_step += 1

        current_lr = scheduler.get_last_lr()[0]
        pbar.set_postfix(loss=float(loss.item() * GRAD_ACCUM), lr=f"{current_lr:.2e}")

    avg_train_loss = train_loss_sum / len(train_loader)

    model.eval()
    val_loss_sum = 0.0
    with torch.no_grad():
        vbar = tqdm(valid_loader, total=len(valid_loader), desc=f"Epoch {epoch+1}/{NUM_EPOCHS} [valid]")
        for batch in vbar:
            batch = {k: v.to(device) for k, v in batch.items()}
            with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=torch.cuda.is_available()):
                outputs = model(**batch)
            val_loss_sum += outputs.loss.item()

    avg_val_loss = val_loss_sum / len(valid_loader)
    print(f"Epoch {epoch+1}: train_loss={avg_train_loss:.4f} | val_loss={avg_val_loss:.4f}")

    if avg_val_loss < best_val_loss:
        best_val_loss = avg_val_loss
        model.save_pretrained(OUTPUT_DIR)
        processor.save_pretrained(OUTPUT_DIR)
        print(f"✅ Best checkpoint saved to: {OUTPUT_DIR}")

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print("best_val_loss =", best_val_loss)


# 추론용 후처리 함수

학습은 **정답 텍스트**로 했지만, 제출은 반드시 `a/b/c/d` 알파벳이어야 합니다.

따라서 생성 결과를 다음 순서로 처리합니다.

1. 모델이 이미 `a/b/c/d`를 출력한 경우 바로 사용
2. 생성 텍스트를 보기 텍스트와 **정규화 후 정확 비교**
3. 포함 관계 비교
4. `SequenceMatcher` 기반 유사도 비교
5. 그래도 실패하면 보기 중 가장 유사한 항목 선택


In [ ]:
def normalize_text(x: str) -> str:
    x = str(x).strip().lower()
    x = re.sub(r"\s+", " ", x)
    x = re.sub(r"["'`]", "", x)
    x = x.replace("개 입니다", "개").replace("개입니까", "개")
    return x.strip()

def extract_letter_or_match(generated_text: str, options: Dict[str, str]) -> str:
    text = normalize_text(generated_text)

    # 1) 알파벳만 출력한 경우
    for pat in [
        r"^([abcd])$",
        r"(?:정답|답|answer)\s*[:：]?\s*([abcd])",
        r"\b([abcd])\b"
    ]:
        m = re.search(pat, text)
        if m:
            return m.group(1)

    norm_options = {k: normalize_text(v) for k, v in options.items()}

    # 2) 완전 일치
    for k, v in norm_options.items():
        if text == v:
            return k

    # 3) 포함 관계
    for k, v in norm_options.items():
        if v and (v in text or text in v):
            return k

    # 4) 유사도 비교
    scores = []
    for k, v in norm_options.items():
        ratio = difflib.SequenceMatcher(None, text, v).ratio()
        scores.append((ratio, k))
    scores.sort(reverse=True)
    return scores[0][1]


# Inference

- best checkpoint 기준 추론
- 질문 + 이미지 → 정답 텍스트 생성
- 생성 텍스트 → 보기 매칭 → 알파벳 출력


In [ ]:
# 필요 시 저장된 best adapter 사용
if os.path.exists(os.path.join(OUTPUT_DIR, "adapter_config.json")):
    print("Using fine-tuned adapter from:", OUTPUT_DIR)
else:
    print("Warning: saved adapter not found. Current in-memory model will be used.")

model.eval()
pred_letters = []
raw_outputs = []

for i in tqdm(range(len(test_ds)), desc="Inference"):
    sample = test_ds[i]

    prompt_text = processor.apply_chat_template(
        sample["prompt_messages"],
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = processor(
        text=[prompt_text],
        images=[sample["image"]],
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=torch.cuda.is_available()):
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                use_cache=True,
                eos_token_id=processor.tokenizer.eos_token_id,
                pad_token_id=processor.tokenizer.pad_token_id,
            )

    prompt_len = inputs["input_ids"].shape[1]
    gen_ids = output_ids[:, prompt_len:]
    gen_text = processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()

    pred_letter = extract_letter_or_match(gen_text, sample["options"])

    raw_outputs.append(gen_text)
    pred_letters.append(pred_letter)

print("prediction distribution:", pd.Series(pred_letters).value_counts().sort_index().to_dict())
print("sample outputs:")
for i in range(min(5, len(raw_outputs))):
    print(i, "=>", raw_outputs[i], "=>", pred_letters[i])


# 제출 파일 생성

`sample_submission.csv` 형식에 맞춰 최종 제출 파일을 생성합니다.

출력 형식:
- `id`
- `answer` : `a`, `b`, `c`, `d` 중 하나


In [ ]:
submission = sample_sub_df.copy()

pred_map = dict(zip(test_df["id"].tolist(), pred_letters))
submission["answer"] = submission["id"].map(pred_map)

# 혹시 누락된 값이 있으면 최빈값 fallback
fallback_letter = all_train_df["answer"].mode().iloc[0]
submission["answer"] = submission["answer"].fillna(fallback_letter)

save_path = os.path.join(BASE_DIR, "submission_qwen25vl_final.csv")
submission.to_csv(save_path, index=False, encoding="utf-8-sig")

print("saved:", save_path)
display(submission.head())


# (선택) 검증 샘플 직접 확인

학습 데이터 일부를 골라 모델 생성 결과가 보기와 어떻게 매칭되는지 확인할 수 있습니다.


In [ ]:
def predict_one(row):
    image = Image.open(row["image_path"]).convert("RGB")
    prompt_messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_INSTRUCT}]},
        {"role": "user", "content": [
            {"type": "image", "image": image},
            {"type": "text", "text": build_user_prompt(
                str(row["question"]), str(row["a"]), str(row["b"]), str(row["c"]), str(row["d"])
            )}
        ]},
    ]
    prompt_text = processor.apply_chat_template(prompt_messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[prompt_text], images=[image], return_tensors="pt").to(device)

    with torch.no_grad():
        with torch.autocast(device_type="cuda", dtype=torch.bfloat16, enabled=torch.cuda.is_available()):
            output_ids = model.generate(
                **inputs,
                max_new_tokens=MAX_NEW_TOKENS,
                do_sample=False,
                eos_token_id=processor.tokenizer.eos_token_id,
                pad_token_id=processor.tokenizer.pad_token_id,
            )
    prompt_len = inputs["input_ids"].shape[1]
    gen_ids = output_ids[:, prompt_len:]
    gen_text = processor.batch_decode(gen_ids, skip_special_tokens=True)[0].strip()
    pred = extract_letter_or_match(gen_text, {"a": row["a"], "b": row["b"], "c": row["c"], "d": row["d"]})
    return gen_text, pred

sample_n = min(5, len(valid_part))
sample_rows = valid_part.sample(sample_n, random_state=SEED)

for _, row in sample_rows.iterrows():
    gen_text, pred = predict_one(row)
    print("=" * 80)
    print("id       :", row["id"])
    print("question :", row["question"])
    print("options  :", {"a": row["a"], "b": row["b"], "c": row["c"], "d": row["d"]})
    print("gold     :", row["answer"], "|", row["answer_text"])
    print("generated:", gen_text)
    print("pred     :", pred)
